In [1]:
# Ячейка 1: Среда и воспроизводимость
import os
import sys
import random
import subprocess
from typing import List, Dict, Optional, Tuple
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Фиксируем SEED
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_seed(42)

# Функция для автоматической установки пакетов
def ensure_package(package_name: str, import_name: Optional[str] = None) -> None:
    target = import_name or package_name
    try:
        __import__(target)
    except ImportError:
        print(f"Устанавливаем пакет: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

# Устанавливаем и импортируем
ensure_package("faiss-cpu", "faiss")
ensure_package("sentence-transformers", "sentence_transformers")
ensure_package("scikit-learn", "sklearn")

import faiss
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import display, Markdown

# Определяем устройство
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

print(f"Устройство: {DEVICE}")
print(f"FAISS доступен: {faiss is not None}")

# Создаем папку для артефактов
os.makedirs("./artifacts", exist_ok=True)
print("Папка './artifacts' создана или уже существует.")

Устройство: cpu
FAISS доступен: True
Папка './artifacts' создана или уже существует.


In [2]:
# Ячейка 2: База знаний по NLP и LLM
documents: List[Dict[str, str]] = [
    {
        "doc_id": "nlp_01",
        "title": "Что такое NLP",
        "text": "Natural Language Processing (NLP) — это область искусственного интеллекта, которая фокусируется на взаимодействии между компьютерами и человеческим языком. Цель NLP — научить компьютеры понимать, интерпретировать и генерировать человеческий язык осмысленным и полезным способом. Примеры задач NLP включают машинный перевод, анализ тональности, распознавание именованных сущностей и создание чат-ботов."
    },
    {
        "doc_id": "nlp_02",
        "title": "Трансформеры",
        "text": "Трансформеры — это архитектура нейронных сетей, которая произвела революцию в NLP. Представленная в статье 'Attention is All You Need' в 2017 году, она основана на механизме внимания, который позволяет модели взвешивать важность разных частей входного текста при обработке. Трансформеры лежат в основе таких мощных моделей, как BERT, GPT и T5."
    },
    {
        "doc_id": "nlp_03",
        "title": "Механизм внимания",
        "text": "Механизм внимания (Attention Mechanism) — ключевой компонент современных NLP-моделей. Он позволяет модели фокусироваться на наиболее релевантных частях входных данных при формировании выхода. В отличие от рекуррентных сетей (RNN), которые обрабатывают текст последовательно, внимание может параллельно оценивать связи между всеми словами в предложении, что значительно ускоряет обучение и улучшает понимание контекста."
    },
    {
        "doc_id": "nlp_04",
        "title": "Большие языковые модели (LLM)",
        "text": "Большие языковые модели (LLM), такие как GPT-4, Claude и Llama, — это нейронные сети с миллиардами параметров, обученные на огромных массивах текстовых данных. Они демонстрируют удивительные способности к обобщению и могут решать широкий круг задач без специального дообучения (zero-shot learning). LLM способны генерировать связный текст, переводить языки, писать код, отвечать на вопросы и многое другое."
    },
    {
        "doc_id": "nlp_05",
        "title": "Токенизация",
        "text": "Токенизация — это процесс разбиения текста на более мелкие единицы, называемые токенами. Токенами могут быть слова, части слов (подслова) или отдельные символы. Это первый и обязательный шаг перед подачей текста в NLP-модель. Современные модели, такие как GPT, часто используют подсловную токенизацию (например, Byte-Pair Encoding, BPE), чтобы эффективно обрабатывать редкие и неизвестные слова."
    },
    {
        "doc_id": "nlp_06",
        "title": "Промпт-инжиниринг",
        "text": "Промпт-инжиниринг — это искусство и наука создания эффективных запросов (промптов) для больших языковых моделей. Хорошо сформулированный промпт может значительно улучшить качество ответа модели, направив её в нужное русло. Методы включают few-shot prompting (предоставление примеров), chain-of-thought (просьба рассуждать по шагам) и указание роли модели."
    }
]

print(f"Количество документов в базе: {len(documents)}")
display(pd.DataFrame(documents)[["doc_id", "title"]])

Количество документов в базе: 6


,doc_id,title
0,nlp_01,Что такое NLP
1,nlp_02,Трансформеры
2,nlp_03,Механизм внимания
3,nlp_04,Большие языковые модели (LLM)
4,nlp_05,Токенизация
5,nlp_06,Промпт-инжиниринг


In [3]:
# Ячейка 3: Чанкинг документов
def chunk_text(text: str, chunk_size: int = 250, overlap: int = 40) -> List[str]:
    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        if end >= text_len:
            break
        start = end - overlap
    return chunks

# Параметры чанкинга (базовые)
CHUNK_SIZE = 250
OVERLAP = 40

chunk_rows = []
for doc in documents:
    chunks = chunk_text(doc["text"], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for i, chunk_text_val in enumerate(chunks):
        chunk_rows.append({
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "chunk_id": f"{doc['doc_id']}_chunk_{i:03d}",
            "chunk_text": chunk_text_val
        })

chunks_df = pd.DataFrame(chunk_rows)
print(f"Всего создано чанков: {len(chunks_df)}")
display(chunks_df.head(3))

Всего создано чанков: 12


,doc_id,title,chunk_id,chunk_text
0,nlp_01,Что такое NLP,nlp_01_chunk_000,Natural Language Processing (NLP) — это област...
1,nlp_01,Что такое NLP,nlp_01_chunk_001,ть и генерировать человеческий язык осмысленны...
2,nlp_02,Трансформеры,nlp_02_chunk_000,Трансформеры — это архитектура нейронных сетей...


In [4]:
# Ячейка 4: Эмбеддинги и индекс FAISS
model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
model = SentenceTransformer(model_name, device=DEVICE)

chunk_texts = chunks_df["chunk_text"].tolist()
print("Вычисляем эмбеддинги для чанков...")
chunk_embeddings = model.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=True)
print(f"Форма матрицы эмбеддингов: {chunk_embeddings.shape}")

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)
print(f"Индекс FAISS создан. Количество векторов в индексе: {index.ntotal}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Вычисляем эмбеддинги для чанков...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Форма матрицы эмбеддингов: (12, 384)
Индекс FAISS создан. Количество векторов в индексе: 12


In [5]:
# Ячейка 5: Функция поиска
def search(query: str, top_k: int = 4) -> pd.DataFrame:
    query_embedding = model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(query_embedding, top_k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk = chunks_df.iloc[idx].to_dict()
        chunk["rank"] = rank
        chunk["score"] = score
        results.append(chunk)

    return pd.DataFrame(results)[["rank", "score", "doc_id", "title", "chunk_id", "chunk_text"]]

# Проверка
sample_query = "Как работает механизм внимания в трансформерах?"
display(Markdown(f"### Запрос: {sample_query}"))
display(search(sample_query, top_k=4))

### Запрос: Как работает механизм внимания в трансформерах?

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.695289,nlp_02,Трансформеры,nlp_02_chunk_000,Трансформеры — это архитектура нейронных сетей...
1,2,0.678464,nlp_03,Механизм внимания,nlp_03_chunk_000,Механизм внимания (Attention Mechanism) — ключ...
2,3,0.411338,nlp_03,Механизм внимания,nlp_03_chunk_001,"рентных сетей (RNN), которые обрабатывают текс..."
3,4,0.403923,nlp_02,Трансформеры,nlp_02_chunk_001,звешивать важность разных частей входного текс...


In [6]:
# Ячейка 6: Контрольные запросы и оценка retrieval
benchmark_queries = [
    {"query_id": "q01", "query": "Что такое NLP?", "relevant_doc_id": "nlp_01"},
    {"query_id": "q02", "query": "Какая архитектура нейросетей произвела революцию в NLP?", "relevant_doc_id": "nlp_02"},
    {"query_id": "q03", "query": "Что позволяет модели взвешивать важность слов?", "relevant_doc_id": "nlp_03"},
    {"query_id": "q04", "query": "Приведи примеры больших языковых моделей.", "relevant_doc_id": "nlp_04"},
    {"query_id": "q05", "query": "Что такое токенизация?", "relevant_doc_id": "nlp_05"},
    {"query_id": "q06", "query": "Как создавать эффективные запросы к LLM?", "relevant_doc_id": "nlp_06"},
]

def evaluate_retrieval(queries, k=4):
    eval_results = []
    for item in queries:
        query = item["query"]
        relevant_id = item["relevant_doc_id"]
        results_df = search(query, top_k=k)

        hit = 1 if relevant_id in results_df["doc_id"].values else 0
        recall = hit
        rank = results_df[results_df["doc_id"] == relevant_id]["rank"].values
        first_rank = rank[0] if len(rank) > 0 else None

        eval_results.append({
            "query_id": item["query_id"],
            "query": query,
            "expected_doc_id": relevant_id,
            "retrieved_doc_ids": ", ".join(results_df["doc_id"].values),
            f"hit@{k}": hit,
            f"recall@{k}": recall,
            "rank_of_first_relevant": first_rank
        })
    return pd.DataFrame(eval_results)

eval_df = evaluate_retrieval(benchmark_queries, k=4)
display(eval_df)

print(f"Средний hit@4: {eval_df['hit@4'].mean():.2f}")
print(f"Средний recall@4: {eval_df['recall@4'].mean():.2f}")

eval_df.to_csv("./artifacts/retrieval_eval.csv", index=False)
print("\nРезультаты сохранены в './artifacts/retrieval_eval.csv'")

,query_id,query,expected_doc_id,retrieved_doc_ids,hit@4,recall@4,rank_of_first_relevant
0,q01,Что такое NLP?,nlp_01,"nlp_01, nlp_03, nlp_02, nlp_05",1,1,1
1,q02,Какая архитектура нейросетей произвела революц...,nlp_02,"nlp_02, nlp_03, nlp_01, nlp_02",1,1,1
2,q03,Что позволяет модели взвешивать важность слов?,nlp_03,"nlp_01, nlp_03, nlp_06, nlp_06",1,1,2
3,q04,Приведи примеры больших языковых моделей.,nlp_04,"nlp_04, nlp_01, nlp_06, nlp_04",1,1,1
4,q05,Что такое токенизация?,nlp_05,"nlp_05, nlp_05, nlp_06, nlp_02",1,1,1
5,q06,Как создавать эффективные запросы к LLM?,nlp_06,"nlp_04, nlp_04, nlp_06, nlp_01",1,1,3


Средний hit@4: 1.00
Средний recall@4: 1.00

Результаты сохранены в './artifacts/retrieval_eval.csv'


In [7]:
# Ячейка 7: Эксперимент с параметром chunk_size
def run_experiment(chunk_size):
    print(f"\n--- Запуск эксперимента с chunk_size={chunk_size} ---")
    temp_chunks = []
    for doc in documents:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=40)
        for i, chunk_text_val in enumerate(chunks):
            temp_chunks.append({
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "chunk_id": f"{doc['doc_id']}_chunk_{i:03d}",
                "chunk_text": chunk_text_val
            })
    temp_chunks_df = pd.DataFrame(temp_chunks)
    temp_texts = temp_chunks_df["chunk_text"].tolist()
    temp_embeddings = model.encode(temp_texts, normalize_embeddings=True, show_progress_bar=False)

    temp_index = faiss.IndexFlatIP(temp_embeddings.shape[1])
    temp_index.add(temp_embeddings)

    global chunks_df, index
    original_chunks_df, original_index = chunks_df, index
    chunks_df, index = temp_chunks_df, temp_index

    eval_df = evaluate_retrieval(benchmark_queries, k=4)

    chunks_df, index = original_chunks_df, original_index

    return {
        "chunk_size": chunk_size,
        "num_chunks": len(temp_chunks_df),
        "mean_hit@4": eval_df['hit@4'].mean(),
        "mean_recall@4": eval_df['recall@4'].mean()
    }

exp_results = [run_experiment(150), run_experiment(350)]
exp_df = pd.DataFrame(exp_results)
display(exp_df)


--- Запуск эксперимента с chunk_size=150 ---

--- Запуск эксперимента с chunk_size=350 ---


,chunk_size,num_chunks,mean_hit@4,mean_recall@4
0,150,22,1.0,1.0
1,350,11,1.0,1.0


In [8]:
# Ячейка 8: Обновление базы знаний и переиндексация
queries_for_comparison = [
    "Что такое RAG в контексте LLM?",
    "Как работает RLHF?",
    "Зачем нужен fine-tuning?"
]

print("=== РЕЗУЛЬТАТЫ ДО ОБНОВЛЕНИЯ ===")
before_results = {}
for q in queries_for_comparison:
    res_df = search(q, top_k=4)
    before_results[q] = ", ".join(res_df["doc_id"].values)
    print(f"\nЗапрос: {q}\nНайденные ID: {before_results[q]}")
    display(res_df[["rank", "score", "doc_id", "chunk_text"]])

# Добавляем новые документы
new_documents = [
    {
        "doc_id": "nlp_07",
        "title": "RAG (Retrieval-Augmented Generation)",
        "text": "RAG (Retrieval-Augmented Generation) — это подход, который объединяет поиск информации и генеративные модели. Сначала система извлекает релевантные документы из внешней базы знаний, а затем языковая модель использует эту информацию для генерации более точного и фактически обоснованного ответа. RAG особенно полезен для задач, требующих актуальных знаний, которые модель могла не видеть во время обучения."
    },
    {
        "doc_id": "nlp_08",
        "title": "RLHF (Reinforcement Learning from Human Feedback)",
        "text": "RLHF (Reinforcement Learning from Human Feedback) — это метод обучения языковых моделей, при котором люди оценивают ответы модели, и эта обратная связь используется для улучшения модели с помощью обучения с подкреплением. RLHF является ключевым компонентом при создании полезных и безопасных ассистентов, таких как ChatGPT. Он помогает согласовать поведение модели с человеческими предпочтениями."
    },
    {
        "doc_id": "nlp_09",
        "title": "Fine-tuning языковых моделей",
        "text": "Fine-tuning (тонкая настройка) — это процесс дополнительного обучения предобученной языковой модели на небольшом специализированном наборе данных. Это позволяет адаптировать модель для конкретной задачи или предметной области, значительно улучшая её производительность без необходимости обучать модель с нуля, что было бы чрезвычайно дорого и долго."
    }
]
documents.extend(new_documents)
print("\n\nДокументы добавлены. Переиндексация...")

# Переиндексация
chunk_rows = []
for doc in documents:
    chunks = chunk_text(doc["text"], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for i, chunk_text_val in enumerate(chunks):
        chunk_rows.append({
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "chunk_id": f"{doc['doc_id']}_chunk_{i:03d}",
            "chunk_text": chunk_text_val
        })
chunks_df = pd.DataFrame(chunk_rows)
chunk_texts = chunks_df["chunk_text"].tolist()
chunk_embeddings = model.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=True)
index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)
print("Переиндексация завершена.")

print("\n=== РЕЗУЛЬТАТЫ ПОСЛЕ ОБНОВЛЕНИЯ ===")
comparison_data = []
for q in queries_for_comparison:
    res_df = search(q, top_k=4)
    after_doc_ids = ", ".join(res_df["doc_id"].values)
    print(f"\nЗапрос: {q}\nНайденные ID: {after_doc_ids}")
    display(res_df[["rank", "score", "doc_id", "chunk_text"]])

    comparison_data.append({
        "query": q,
        "before_retrieved_doc_ids": before_results[q],
        "after_retrieved_doc_ids": after_doc_ids,
        "changed": before_results[q] != after_doc_ids
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df.to_csv("./artifacts/retrieval_before_after_update.csv", index=False)
print("\nРезультаты сравнения сохранены в './artifacts/retrieval_before_after_update.csv'")
display(comparison_df)

=== РЕЗУЛЬТАТЫ ДО ОБНОВЛЕНИЯ ===

Запрос: Что такое RAG в контексте LLM?
Найденные ID: nlp_04, nlp_04, nlp_03, nlp_05


,rank,score,doc_id,chunk_text
0,1,0.450056,nlp_04,ению и могут решать широкий круг задач без спе...
1,2,0.440788,nlp_04,"Большие языковые модели (LLM), такие как GPT-4..."
2,3,0.418230,nlp_03,Механизм внимания (Attention Mechanism) — ключ...
3,4,0.331113,nlp_05,Токенизация — это процесс разбиения текста на ...



Запрос: Как работает RLHF?
Найденные ID: nlp_04, nlp_04, nlp_03, nlp_02


,rank,score,doc_id,chunk_text
0,1,0.342621,nlp_04,ению и могут решать широкий круг задач без спе...
1,2,0.288998,nlp_04,"Большие языковые модели (LLM), такие как GPT-4..."
2,3,0.288246,nlp_03,Механизм внимания (Attention Mechanism) — ключ...
3,4,0.237787,nlp_02,звешивать важность разных частей входного текс...



Запрос: Зачем нужен fine-tuning?
Найденные ID: nlp_06, nlp_06, nlp_03, nlp_02


...

,rank,score,doc_id,chunk_text
0,1,0.667864,nlp_08,RLHF (Reinforcement Learning from Human Feedba...
1,2,0.559170,nlp_08,креплением. RLHF является ключевым компонентом...
2,3,0.342621,nlp_04,ению и могут решать широкий круг задач без спе...
3,4,0.288998,nlp_04,"Большие языковые модели (LLM), такие как GPT-4..."



Запрос: Зачем нужен fine-tuning?
Найденные ID: nlp_09, nlp_07, nlp_06, nlp_06


,rank,score,doc_id,chunk_text
0,1,0.578610,nlp_09,Fine-tuning (тонкая настройка) — это процесс д...
1,2,0.447913,nlp_07,льзует эту информацию для генерации более точн...
2,3,0.421789,nlp_06,Промпт-инжиниринг — это искусство и наука созд...
3,4,0.410136,nlp_06,ужное русло. Методы включают few-shot promptin...



Результаты сравнения сохранены в './artifacts/retrieval_before_after_update.csv'


,query,before_retrieved_doc_ids,after_retrieved_doc_ids,changed
0,Что такое RAG в контексте LLM?,"nlp_04, nlp_04, nlp_03, nlp_05","nlp_07, nlp_07, nlp_04, nlp_04",True
1,Как работает RLHF?,"nlp_04, nlp_04, nlp_03, nlp_02","nlp_08, nlp_08, nlp_04, nlp_04",True
2,Зачем нужен fine-tuning?,"nlp_06, nlp_06, nlp_03, nlp_02","nlp_09, nlp_07, nlp_06, nlp_06",True


In [9]:
# Ячейка 9: Mini-RAG (с другим extractive-подходом)
def split_into_sentences(text):
    return re.split(r'(?<=[.!?])\s+', text)

def mini_rag_answer(query: str, top_k: int = 4) -> Dict:
    retrieved_df = search(query, top_k=top_k)

    # Собираем контекст
    context = "\n\n".join([f"[{row['doc_id']}] {row['chunk_text']}" for _, row in retrieved_df.iterrows()])

    # ОТЛИЧИЕ: Берем ПОСЛЕДНЕЕ предложение из самого релевантного чанка
    top_chunk_text = retrieved_df.iloc[0]['chunk_text']
    sentences = split_into_sentences(top_chunk_text)
    answer_sentence = sentences[-1] if sentences else top_chunk_text

    return {
        "question": query,
        "answer": answer_sentence,
        "retrieved_sources": ", ".join(retrieved_df["doc_id"].unique()),
        "context": context
    }

# Примеры
rag_examples = []
test_questions = [
    "Что такое NLP?",
    "Как работают трансформеры?",
    "Как улучшить промпт для LLM?"
]

for q in test_questions:
    rag_result = mini_rag_answer(q)
    print(f"Вопрос: {q}")
    print(f"Ответ: {rag_result['answer']}")
    print(f"Источники: {rag_result['retrieved_sources']}")
    print("-" * 50)

    rag_examples.append({
        "question": q,
        "answer": rag_result['answer'],
        "retrieved_sources": rag_result['retrieved_sources']
    })

rag_examples_df = pd.DataFrame(rag_examples)
rag_examples_df.to_csv("./artifacts/rag_examples.csv", index=False)
print("Примеры RAG сохранены в './artifacts/rag_examples.csv'")

Вопрос: Что такое NLP?
Ответ: Цель NLP — научить компьютеры понимать, интерпретировать и генерировать человеческий язык осмы
Источники: nlp_01, nlp_03, nlp_02, nlp_05
--------------------------------------------------
Вопрос: Как работают трансформеры?
Ответ: Представленная в статье 'Attention is All You Need' в 2017 году, она основана на механизме внимания, который позволяет модели взвешивать важность разных частей входног
Источники: nlp_02, nlp_06, nlp_04
--------------------------------------------------
Вопрос: Как улучшить промпт для LLM?
Ответ: LLM способны генерировать связный текст, переводить языки, писать код, отвечать на вопросы и многое другое.
Источники: nlp_04, nlp_06, nlp_08
--------------------------------------------------
Примеры RAG сохранены в './artifacts/rag_examples.csv'
